# 16 — Oportunidade comercial

O classificador BERTimbau ajustado na Sprint 3 é reutilizado quando o checkpoint local existe. A saída binária é `detectada` ou `nao_detectada` e sempre permanece sujeita à revisão humana.

## Checkpoint e agregação

O checkpoint não é versionado. Ele é carregado sob demanda de `data/processed/bertimbau_opportunity_best`, usa GPU quando disponível e agrega a evidência dos chunks da transcrição.

In [ ]:
OPPORTUNITY_MODEL_PATH = "data/processed/bertimbau_opportunity_best"

_OPPORTUNITY_CLASSIFIER = None

_OPPORTUNITY_LOAD_ERROR = None


## Inferência por chunks

Cada chunk respeita o limite do BERTimbau. Para uma transcrição curta, uma evidência forte é suficiente; em textos com vários chunks, são exigidos ao menos dois sinais com probabilidade mínima de 0,80 e densidade de 5%. O score final resume os três chunks mais fortes.

In [ ]:
def _load_opportunity_classifier():
    global _OPPORTUNITY_CLASSIFIER, _OPPORTUNITY_LOAD_ERROR
    if _OPPORTUNITY_CLASSIFIER is not None:
        return _OPPORTUNITY_CLASSIFIER
    if _OPPORTUNITY_LOAD_ERROR is not None:
        raise RuntimeError("Checkpoint BERTimbau indisponível.") from _OPPORTUNITY_LOAD_ERROR
    try:
        checkpoint = _project_root() / OPPORTUNITY_MODEL_PATH
        if not checkpoint.exists():
            raise FileNotFoundError(f"Checkpoint não encontrado: {checkpoint}")
        from transformers import pipeline

        device = -1
        try:
            import torch

            if torch.cuda.is_available():
                device = 0
        except ImportError:
            pass
        _OPPORTUNITY_CLASSIFIER = pipeline(
            "text-classification",
            model=str(checkpoint),
            tokenizer=str(checkpoint),
            device=device,
        )
        return _OPPORTUNITY_CLASSIFIER
    except Exception as error:
        _OPPORTUNITY_LOAD_ERROR = error
        raise RuntimeError("Checkpoint BERTimbau indisponível.") from error

def _opportunity_probability(output: Any) -> float:
    if isinstance(output, list) and output and isinstance(output[0], list):
        output = output[0]
    if not isinstance(output, list):
        raise ValueError("Saída inválida do classificador de oportunidade.")
    for item in output:
        label = str(item.get("label", "")).casefold()
        if label in {"oportunidade", "label_1", "1"}:
            return float(item["score"])
    raise ValueError("O classificador não devolveu a classe oportunidade.")

def _model_opportunity(transcription: str) -> dict[str, Any]:
    classifier = _load_opportunity_classifier()
    probabilities = [
        _opportunity_probability(
            classifier(chunk, truncation=True, max_length=512, top_k=None)
        )
        for chunk in _word_chunks(transcription, max_words=320)
    ]
    supporting = sum(probability >= 0.80 for probability in probabilities)
    required_support = 1 if len(probabilities) == 1 else 2
    density = supporting / len(probabilities)
    top_probabilities = sorted(probabilities, reverse=True)[:3]
    score = sum(top_probabilities) / len(top_probabilities)
    detected = supporting >= required_support and density >= 0.05
    return {
        "label": "detectada" if detected else "nao_detectada",
        "score": round(score, 6),
        "score_type": "model_probability",
        "engine": "bertimbau_finetuned",
        "model": OPPORTUNITY_MODEL_PATH,
    }


## Fallback comercial

As regras exigem combinação de intenção ou compra com contexto de dor ou produto. Recusas explícitas prevalecem, reduzindo falsos sinais de oportunidade.

In [ ]:
OPPORTUNITY_INTENT_SIGNALS = {
    "precisamos", "preciso", "necessitamos", "queremos",
    "gostariamos", "buscamos", "procuramos", "avaliar",
    "avaliando", "interesse", "interessado", "interessados",
}

OPPORTUNITY_BUY_SIGNALS = {
    "proposta", "cotacao", "orcamento", "preco", "valor",
    "contratar", "adquirir", "comprar", "implantacao",
    "implantar", "implementar", "substituir", "prazo",
}

OPPORTUNITY_PAIN_SIGNALS = {
    "problema", "dificuldade", "gargalo", "retrabalho", "manual",
    "planilha", "integracao", "lentidao", "erro", "nao atende",
    "limitacao",
}

OPPORTUNITY_PRODUCT_SIGNALS = {
    "erp", "crm", "software", "sistema", "solucao", "plataforma",
    "protheus", "fluig", "datasul", "totvs rm", "hcm", "wms",
    "licenca", "modulo",
}

OPPORTUNITY_REFUSAL_SIGNALS = {
    "sem interesse", "nao temos interesse", "nao precisamos",
    "nao pretendemos", "nao vamos contratar", "projeto cancelado",
    "projeto descartado",
}


In [ ]:
def _signal_hits(normalized: str, signals: set[str]) -> set[str]:
    return {
        signal
        for signal in signals
        if re.search(rf"(?<!\w){re.escape(signal)}(?!\w)", normalized)
    }

def _lexical_opportunity(transcription: str) -> dict[str, Any]:
    normalized = _normalize(transcription)
    refusals = _signal_hits(normalized, OPPORTUNITY_REFUSAL_SIGNALS)
    if refusals:
        return {
            "label": "nao_detectada",
            "score": 1.0,
            "score_type": "heuristic",
            "engine": "lexical_opportunity",
            "model": None,
        }

    intent = _signal_hits(normalized, OPPORTUNITY_INTENT_SIGNALS)
    buying = _signal_hits(normalized, OPPORTUNITY_BUY_SIGNALS)
    pain = _signal_hits(normalized, OPPORTUNITY_PAIN_SIGNALS)
    product = _signal_hits(normalized, OPPORTUNITY_PRODUCT_SIGNALS)
    evidence_score = 2 * bool(intent) + 2 * bool(buying) + bool(pain) + bool(product)
    detected = evidence_score >= 4 and bool(intent or buying)
    return {
        "label": "detectada" if detected else "nao_detectada",
        "score": round(evidence_score / 6.0, 6),
        "score_type": "heuristic",
        "engine": "lexical_opportunity",
        "model": None,
    }


## Modelo ou fallback

No modo automático, a ausência do artefato local não interrompe a análise; o integrador recebe o motivo técnico do fallback sem expor a transcrição.

In [ ]:
def _analyze_opportunity(transcription: str, mode: str) -> tuple[dict[str, Any], str, str | None]:
    if mode == "fallback":
        return _lexical_opportunity(transcription), "fallback", None
    try:
        return _model_opportunity(transcription), "model", None
    except Exception as error:
        if mode == "full":
            raise RuntimeError("O modo full exige o checkpoint BERTimbau.") from error
        return _lexical_opportunity(transcription), "fallback", type(error).__name__
